# M2 v2 · Evaluación en prueba (una sola vez)

**TFM · HAM10000** · Responsable: **Alex Reinoso**.

Evalúa las tres semillas con alias `final` de la selección v2. Para cada red hace **una** pasada por el conjunto de prueba: argmax y la regla `p(mel) ≥ umbral_mel`. Registra `test_*`, `test_umbral_*`, las dos matrices normalizadas y `predicciones_test.csv`.

`umbral_mel` tiene que estar ya como parámetro del run `final`. Si el run `m2-test-<arch>` tiene `test_evaluado=true`, el código se niega a repetirlo.

No lances esta celda hasta que la selección v2 esté en `main` y tú confirmes que se abre la prueba.


## 0. Parámetros

In [ ]:
TAG = "main"          # el commit que contiene src/evaluation.py
AUTOR = "alex"
CONFIRMO_TEST = False   # pasarlo a True solo cuando se decida abrir la prueba
ARCHS = ["resnet50", "densenet121", "efficientnetb0"]


## 1. Sesión

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU"
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {TAG}
!git pull -q origin {TAG}
!pip install -q -r requirements.txt


In [ ]:
import os, shutil, zipfile, time
from pathlib import Path
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
assert CONFIRMO_TEST, "CONFIRMO_TEST está en False. No se abre el conjunto de prueba."
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not (LOCAL / "images").exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido ({time.time()-t0:.0f} s)")
assert len(list((LOCAL / "images").glob("*.jpg"))) == 10015
assert file_sha256("data/splits/split_v1.csv") == "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"
print("[+] Split v1 verificado")


In [ ]:
import os
import mlflow
from google.colab import userdata
from src.tracking import setup_mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
print(setup_mlflow(cfg))


## 2. Evaluar\nUna arquitectura cada vez. Si una ya está marcada, el bucle se detiene y no toca las demás.

In [ ]:
assert CONFIRMO_TEST, "CONFIRMO_TEST está en False. No se abre el conjunto de prueba."
from src.evaluation import lanzar_evaluacion_test
for arch in ARCHS:
    print(f"\n{'='*20} {arch} {'='*20}")
    r = lanzar_evaluacion_test(arch, cfg, autor=AUTOR)
    print(f"[+] {arch} run={r['run_id']} umbral_mel={r['umbral_mel']:.4f} "
          f"test_sens_mel={r['test_mel_sensitivity']:.3f} "
          f"test_umbral_sens_mel={r['test_umbral_mel_sensitivity']:.3f}")
